### Tools

Models can request to call tools that perform tasks such as fetching data from database, searching the web, or running code. Tools are pairings of :

   1. A schema , including the name of the tool, a description, and/or argument definition(often a JSON schema)
   2. A function to execute

In [1]:
import os

from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-safeguard-20b")

response = model.invoke("Why do parrots talk?")
response

AIMessage(content='**Short answer:**  \nParrots talk because they are *vocal learners*—they have a specialized vocal organ (the syrinx) and a brain structure that lets them imitate sounds. In the wild this ability helps them communicate with their flock, but when kept by humans it often turns into “speech” that we can understand.\n\n---\n\n## 1. The biology that makes talking possible\n\n| Feature | What it does | Why it matters for parrots |\n|---------|--------------|----------------------------|\n| **Syrinx** | The “vocal organ” at the base of the trachea. | Allows a wide range of tones, including those that mimic human speech. |\n| **Large brain relative to body** | Especially a developed *telencephalon* and *pallial* regions. | Supports complex sound processing and memory. |\n| **Vocal learning circuitry** | Neural pathways similar to those used in human speech. | Enables parrots to *learn* and *repeat* sounds they hear. |\n| **Social brain** | Strong need for group cohesion. | Mi

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f'Tool: {tool_call['name']}')
    print(f'Args: {tool_call['args']}')

content='' additional_kwargs={'reasoning_content': 'We need to use the get_weather function. We\'ll call it with location "Boston".', 'tool_calls': [{'id': 'fc_fd6b328e-b580-4614-abea-83d4cedca2e3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 127, 'total_tokens': 168, 'completion_time': 0.043795315, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.015767598, 'prompt_tokens_details': None, 'queue_time': 0.198482319, 'total_time': 0.059562913}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a09178-c1c2-7650-aebc-8bee5ebdd6eb-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_fd6b328e-b580-4614-abea-83d4cedca2e3', 'type': 'tool_call'}] invalid_tool_calls=[

### Tool Execution Loops

In [5]:
### 1. Model generates tool calls
messages =[{"role": "user", "content": "What is the weather like in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


### 2. Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


### 3. Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)

content='It’s sunny in Boston.' additional_kwargs={'reasoning_content': 'We need to respond: "It\'s sunny in Boston". Probably add context.'} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 157, 'total_tokens': 188, 'completion_time': 0.030977752, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.007624105, 'prompt_tokens_details': None, 'queue_time': 0.00279025, 'total_time': 0.038601857}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a09182-3515-7043-8df0-9a38a8d1b623-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 157, 'output_tokens': 31, 'total_tokens': 188, 'output_token_details': {'reasoning': 16}}


In [6]:
messages

[{'role': 'user', 'content': 'What is the weather like in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather like in Boston?" We can use the get_weather function. We\'ll call it with location "Boston". Then output the result. We\'ll need to call the function.', 'tool_calls': [{'id': 'fc_f326a285-b432-4430-a92d-d654fcecce8d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 128, 'total_tokens': 193, 'completion_time': 0.06504327, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.010584445, 'prompt_tokens_details': None, 'queue_time': 0.023373076, 'total_time': 0.075627715}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09182-2